# RAG 1: bardzo prosty przykład

Ten notebook pokazuje **najprostszy możliwy Retrieval-Augmented Generation**:
1. mamy mały zbiór dokumentów,
2. wyszukujemy najbardziej podobne fragmenty metodą **TF-IDF + cosine similarity**,
3. budujemy odpowiedź na podstawie odnalezionych źródeł.

To jest wersja **offline**, bez zewnętrznego API.

## Cel dydaktyczny
Ten przykład dobrze nadaje się na początek zajęć z NLP, bo pokazuje samą ideę RAG bez ciężkiej infrastruktury:
- **retrieval**: znajdź pasujące dokumenty,
- **augmentation**: dołącz kontekst,
- **generation**: wygeneruj odpowiedź na podstawie kontekstu.

In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

In [2]:
documents = [
    {
        "id": "doc1",
        "title": "Transformery w NLP",
        "text": "Transformery to architektura sieci neuronowych oparta na mechanizmie self-attention. "
                "W NLP są używane do tłumaczenia, klasyfikacji tekstu, streszczania i generowania odpowiedzi."
    },
    {
        "id": "doc2",
        "title": "Embeddingi",
        "text": "Embeddingi reprezentują słowa, zdania lub dokumenty jako wektory liczbowe. "
                "Pozwalają porównywać podobieństwo semantyczne między tekstami."
    },
    {
        "id": "doc3",
        "title": "Czym jest RAG",
        "text": "RAG, czyli Retrieval-Augmented Generation, łączy wyszukiwanie informacji z generowaniem odpowiedzi. "
                "System najpierw pobiera istotne fragmenty wiedzy, a potem tworzy odpowiedź z użyciem tego kontekstu."
    },
    {
        "id": "doc4",
        "title": "Tokenizacja",
        "text": "Tokenizacja polega na dzieleniu tekstu na mniejsze jednostki, takie jak słowa, subwordy lub znaki. "
                "To jeden z pierwszych etapów przetwarzania języka naturalnego."
    }
]

df_docs = pd.DataFrame(documents)
df_docs

,id,title,text
0,doc1,Transformery w NLP,Transformery to architektura sieci neuronowych...
1,doc2,Embeddingi,"Embeddingi reprezentują słowa, zdania lub doku..."
2,doc3,Czym jest RAG,"RAG, czyli Retrieval-Augmented Generation, łąc..."
3,doc4,Tokenizacja,Tokenizacja polega na dzieleniu tekstu na mnie...


In [3]:
vectorizer = TfidfVectorizer()
doc_matrix = vectorizer.fit_transform(df_docs["text"])

def retrieve(query, top_k=2):
    query_vec = vectorizer.transform([query])
    sims = cosine_similarity(query_vec, doc_matrix).flatten()
    best_idx = sims.argsort()[::-1][:top_k]
    results = []
    for idx in best_idx:
        results.append({
            "id": df_docs.iloc[idx]["id"],
            "title": df_docs.iloc[idx]["title"],
            "text": df_docs.iloc[idx]["text"],
            "score": float(sims[idx])
        })
    return results

In [4]:
def generate_answer(query, retrieved_docs):
    context = " ".join(doc["text"] for doc in retrieved_docs)
    answer = (
        f"Pytanie: {query}\n\n"
        f"Odpowiedź na podstawie odnalezionych dokumentów:\n"
        f"{context}\n\n"
        f"Źródła: {', '.join(doc['id'] for doc in retrieved_docs)}"
    )
    return answer

In [5]:
query = "Na czym polega RAG w NLP?"
retrieved = retrieve(query, top_k=2)

for r in retrieved:
    print(f"{r['id']} | {r['title']} | score={r['score']:.4f}")

doc4 | Tokenizacja | score=0.2638
doc1 | Transformery w NLP | score=0.1982


In [6]:
print(generate_answer(query, retrieved))

Pytanie: Na czym polega RAG w NLP?

Odpowiedź na podstawie odnalezionych dokumentów:
Tokenizacja polega na dzieleniu tekstu na mniejsze jednostki, takie jak słowa, subwordy lub znaki. To jeden z pierwszych etapów przetwarzania języka naturalnego. Transformery to architektura sieci neuronowych oparta na mechanizmie self-attention. W NLP są używane do tłumaczenia, klasyfikacji tekstu, streszczania i generowania odpowiedzi.

Źródła: doc4, doc1


## Wnioski
Ten wariant jest bardzo prosty, ale pokazuje serce całego pomysłu:
- najpierw wyszukiwanie,
- potem odpowiedź oparta o znaleziony kontekst.

W praktyce taki generator można potem podmienić na model LLM.